# 10 — Codebook v1.0: ratification, freeze, and assignment (Track-1 map, Track-2 placed into it)

**Purpose.** Apply the delegated ratification to the 150-topic draft (09),
freeze the ratified codebook (SHA-256), then assign: Track-1 phrases by
**direct cluster membership**, Track-2 stance-record topics by **nearest
ratified-topic centroid** with a cosine floor — the layered architecture:
Track 1 draws the hearing-level map, Track 2's utterance-level positions
are placed onto it, and Track-2 phrases below the floor stay in the other
pool as unplaced fine-grained detail.

**Ratification summary** (every decision with its reason in the RATIFY
tables; full record in `quality_reports/20260822_codebook_ratification.md`):
KEEP 111, MERGE 9, DROP 30. Drops are surface-form families (named-Act /
"program" / "initiative" / "reduction" word clusters), heterogeneous
grab-bags (low stability *and* low cohesion *and* mixed members), and
sub-4-phrase fragments. Stability and cohesion are carried into the
codebook as reported diagnostics.

**Inputs.** `taxonomy/` artifacts (08, 09); Track-2 `stance_records` (05).

**Outputs.** `taxonomy/codebook_v1.parquet` (frozen),
`track1_assigned.parquet`, `track2_assigned.parquet`,
`hearing_topics_v1.parquet`, ratification record.

## Ratification decisions

In [1]:
import hashlib
import json as _json
import os
import sys

sys.path.insert(0, "../lib")

import numpy as np
import pandas as pd

from paths import HF_CACHE, ROOT, topics_run_dir

os.environ["HF_HOME"] = str(HF_CACHE)
K_TOPIC, K_THEME = 150, 15
TAX = topics_run_dir("taxonomy")
draft = pd.read_parquet(TAX / "codebook_draft.parquet")
topics_d = draft[draft["kind"] == "topic"].set_index("cluster")
themes_d = draft[draft["kind"] == "theme"].set_index("cluster")

DROP = {
 104: "surface family of named Acts across domains (Jones/GLBA/FAST/FIRST)",
 82: "legal-clause family (First Amendment/grandfather/Section 13(b))",
 88: "grab-bag: auctions+MOU+pricing (s0.24, c0.55)",
 96: "'standards' word family across domains (NIST/CAFE/forensic)",
 97: "grab-bag: Yucca+data brokers+Form 477",
 98: "heterogeneous internet terms (interoperability/IoT/ICANN)",
 99: "census + cost-benefit mixture",
 123: "'program' word family (BEAD/I-Corps/NPOESS/TWIC)",
 70: "'partnership' word family (PPP/Bayh-Dole/Screening)",
 72: "'initiative' word family (Lands Legacy/SunShot/BRAIN)",
 124: "'assistance program' mixture (E-Rate/ACP/Visa Waiver/Paycheck)",
 57: "'America act' surface family (Connect/GROW/Brand USA/Jobs Plan)",
 28: "'reduction' word family (IRA/GHG/Deficit Reduction)",
 71: "planning-office fragments (JPDO), incoherent remainder",
 113: "research-program mixture (Low Dose/CRS/Engineering), s0.27",
 51: "mixed telephone/alert terms",
 77: "regulatory-framework grab-bag (codes/Fast Track/NGB/OIRA)",
 85: "tax/carbon mixture (RGGI/social cost/gas tax/HMT)",
 80: "licensing-compliance grab-bag (retransmission/APA/BOD), s0.28",
 81: "incoherent mixture (cross-ownership/quarantine/ADS-B)",
 106: "incoherent mixture (FISMA/ADS-B/Atlas V), s0.25 c0.54",
 140: "'-Gen' surface family (NextGen aviation + FutureGen coal)",
 75: "'reform' word family (immigration/CONCACAF/CPB/MAP-21)",
 63: "1 phrase", 131: "2 phrases", 66: "2 phrases", 135: "3 junk phrases",
 39: "3 junk phrases (Cuba aid/Katrina)", 101: "5 mixed phrases",
 142: "5 mixed phrases",
}
MERGE = {
 105: (32, "flow valves/blowout preventers = pipeline safety equipment"),
 33: (78, "Clean Air/Water Act phrases belong with EPA regulation"),
 64: (65, "SAFE DATA/WEB acts join the data-protection/spam cluster"),
 8: (6, "ABI is a GOES instrument -> weather-satellite topic"),
 5: (76, "boxing/sports acts join the sports-integrity cluster"),
 31: (44, "Magnuson-Stevens act family joins fisheries management"),
 42: (41, "firefighter grants join fire/emergency response"),
 16: (121, "Noyce teacher scholarships join STEM education"),
 118: (95, "NITRD joins advanced-computing programs"),
}
RELABEL = {
 74: "inspectors general and federal accountability",
 86: "passenger fees and charges",
 95: "advanced computing and ARPA-E programs",
 108: "stem cell research and bioethics",
 116: "science competitiveness legislation",
 133: "surface transportation agencies",
 114: "science policy bodies (OSTP and advisory boards)",
 138: "public alert and risk information systems",
}
KEEP = [c for c in topics_d.index if c not in DROP and c not in MERGE]
assert len(KEEP) + len(DROP) + len(MERGE) == 150
print(f"KEEP {len(KEEP)} / MERGE {len(MERGE)} / DROP {len(DROP)}")

KEEP 111 / MERGE 9 / DROP 30


## Build and freeze codebook v1.0

Topic → theme by phrase-weight plurality at the K=15 cut; themes with no
surviving topic are dropped. Exemplars topped up from members (verbatim,
capped by cluster size).

In [2]:
phrases = pd.read_parquet(TAX / "phrases.parquet")
cuts = pd.read_parquet(TAX / "cut_labels.parquet")
ph = phrases[phrases["family_id"] >= 0].merge(
    cuts[["family_id", f"K{K_TOPIC}", f"K{K_THEME}"]], on="family_id")
ph = ph.rename(columns={f"K{K_TOPIC}": "topic_c", f"K{K_THEME}": "theme_c"})
remap = {c: c for c in KEEP}
remap.update({c: t for c, (t, _) in MERGE.items()})
ph["topic_id"] = ph["topic_c"].map(lambda c: remap.get(c, -1))

nest = (ph[ph["topic_id"] >= 0].groupby(["topic_id", "theme_c"])["weight"]
        .sum().reset_index().sort_values("weight", ascending=False)
        .drop_duplicates("topic_id").set_index("topic_id")["theme_c"])

rows = []
for cid in KEEP:
    r = topics_d.loc[cid]
    e = _json.loads(r["entry"])
    members = _json.loads(r["top_members"])
    ex = list(dict.fromkeys(e["exemplars"]))
    for m in members:
        if len(ex) >= 5:
            break
        if m.casefold() not in {x.casefold() for x in ex}:
            ex.append(m)
    th = int(nest.loc[cid])
    te = (_json.loads(themes_d.loc[th, "entry"])
          if themes_d.loc[th, "entry"] else None)
    rows.append({
        "topic_id": int(cid), "theme_id": th,
        "theme_label": te["label"] if te else f"theme {th}",
        "label": RELABEL.get(cid, e["label"]),
        "definition": e["definition"],
        "exclusion_notes": _json.dumps(e["exclusion_notes"]),
        "exemplars": _json.dumps(ex[:5]),
        "stability": float(r["stability"]), "cohesion": float(r["cohesion"]),
        "weight": float(r["weight"]), "n_phrases": int(r["n_phrases"]),
    })
codebook = pd.DataFrame(rows).sort_values(["theme_id", "topic_id"])
codebook.to_parquet(TAX / "codebook_v1.parquet", index=False)
sha = hashlib.sha256((TAX / "codebook_v1.parquet").read_bytes()).hexdigest()
print(f"codebook v1.0: {len(codebook)} topics / "
      f"{codebook['theme_id'].nunique()} themes")
print(f"FREEZE sha256: {sha}")

codebook v1.0: 111 topics / 14 themes
FREEZE sha256: fd81e7814be416617cb52ac4bfaee3eec8086192aa51fda171342b9bb1919abc


## Assignment

Track-1: direct membership through the family map. Track-2: each record's
topic phrase is embedded and placed at the nearest ratified-topic centroid
(centroids = weight-averaged Track-1 member embeddings) when cosine ≥
τ = 0.68 — the floor validated by the v0 band audit; below it the record's
fine-grained object stays unplaced (other). The floor band is sampled and
printed for the audit trail.

In [3]:
import torch
from transformers import AutoModel, AutoTokenizer

import relevance as R

meta = codebook.set_index("topic_id")
key2topic = ph.set_index("key")["topic_id"]

t1_raw = pd.read_parquet(topics_run_dir("track1") / "topics.parquet")
t1_rows = []
for pkg, year, chamber, js in zip(t1_raw["packageId"], t1_raw["year"],
                                  t1_raw["chamber"], t1_raw["topics_json"]):
    for t in _json.loads(js):
        t1_rows.append({"packageId": pkg, "year": year, "chamber": chamber,
                        "rank": t["rank"], "phrase": t["topic"],
                        "key": t["topic"].casefold().strip()})
t1 = pd.DataFrame(t1_rows)
t1["topic_id"] = t1["key"].map(key2topic).fillna(-1).astype(int)
t1["topic_label"] = t1["topic_id"].map(meta["label"]).fillna("other")
t1.drop(columns=["key"]).to_parquet(TAX / "track1_assigned.parquet",
                                    index=False)

# topic centroids from Track-1 members
dat = np.load(TAX / "unit_embeddings.npz", allow_pickle=True)
X = dat["X"].astype(np.float32)
nk = {n: i for i, n in enumerate(dat["norm_key"])}
ph["ux"] = ph["norm"].str.casefold().map(nk)
tids = sorted(meta.index)
C = np.zeros((len(tids), X.shape[1]), dtype=np.float32)
for k, tid in enumerate(tids):
    g = ph[ph["topic_id"] == tid]
    v = (X[g["ux"].to_numpy()] * g["weight"].to_numpy()[:, None]).sum(0)
    C[k] = v / np.linalg.norm(v)

TAU = 0.68
t2 = pd.read_parquet(topics_run_dir("track2") / "stance_records.parquet")
uniq2 = t2["topic"].str.casefold().str.strip().drop_duplicates()
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(R.ENCODER)
enc = AutoModel.from_pretrained(R.ENCODER, dtype="auto",
                                device_map=dev).eval()
with torch.no_grad():
    E2 = R.encode(uniq2.tolist(), enc, tok, R.ENCODER).astype(np.float32)
S = E2 @ C.T
best = S.argmax(1)
sim = S.max(1)
u2 = pd.DataFrame({"key": uniq2.to_numpy(), "sim": sim,
                   "topic_id": np.where(sim >= TAU,
                                        np.array(tids)[best], -1)})
t2["key"] = t2["topic"].str.casefold().str.strip()
t2 = t2.merge(u2, on="key", how="left")
t2["topic_label"] = t2["topic_id"].map(meta["label"]).fillna("other")
t2.drop(columns=["key"]).to_parquet(TAX / "track2_assigned.parquet",
                                    index=False)

print(f"track1 mentions in codebook topics: "
      f"{(t1['topic_id'] >= 0).mean():.1%}")
print(f"track2 records placed (cos >= {TAU}): "
      f"{(t2['topic_id'] >= 0).mean():.1%}; median sim of placed "
      f"{t2.loc[t2['topic_id'] >= 0, 'sim'].median():.2f}")
band = u2[(u2["sim"] >= TAU) & (u2["sim"] < TAU + 0.04)].sample(
    8, random_state=1089)
print("\nfloor-band audit (0.68-0.72):")
for r in band.itertuples():
    print(f"  {r.sim:.2f}  {r.key[:48]:48s} -> "
          f"{meta.loc[r.topic_id, 'label'][:44]}")

counts = (t1[t1["topic_id"] >= 0].groupby(["packageId", "topic_id"])
          .size().rename("n").reset_index())
modal = (counts.sort_values(["packageId", "n"], ascending=[True, False])
         .drop_duplicates("packageId"))
modal["primary_label"] = modal["topic_id"].map(meta["label"])
modal["primary_theme"] = modal["topic_id"].map(meta["theme_label"])
modal.to_parquet(TAX / "hearing_topics_v1.parquet", index=False)
print(f"\nhearings with a primary topic: "
      f"{modal['packageId'].nunique():,} / "
      f"{t1['packageId'].nunique():,}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

track1 mentions in codebook topics: 83.2%
track2 records placed (cos >= 0.68): 92.6%; median sim of placed 0.78

floor-band audit (0.68-0.72):
  0.69  feedback to children                             -> Children's online privacy and safety legisla
  0.72  atp and mep zeroing out                          -> stem cell research and bioethics
  0.68  nasa inspector general robert cobb's misconduct  -> inspectors general and federal accountabilit
  0.69  last paragraph of the amendment                  -> hours-of-service regulations
  0.70  ifq decisionmaking process                       -> inspectors general and federal accountabilit
  0.69  the idea                                         -> Online advertising and consumer choice polic
  0.71  company's financial and political activities     -> Consumer credit protection and loan guarante
  0.69  software for the 737 max 8s and max 9s           -> Aviation safety and certification policy

hearings with a primary topic: 2,342 / 2,352


## Ratification record

In [4]:
lines = ["# Codebook v1.0 (Track-1 map) — ratified and frozen "
         "(2026-08-22)", "",
         f"Freeze SHA-256: `{sha}`", "",
         f"KEEP {len(KEEP)} / MERGE {len(MERGE)} / DROP {len(DROP)}; "
         "stability and cohesion are reported diagnostics (no gate). "
         "Ratification delegated to the assistant; decisions below. "
         "CAP untouched.", "", "## Drops", ""]
lines += [f"- Topic {c}: {r}" for c, r in sorted(DROP.items())]
lines += ["", "## Merges", ""]
lines += [f"- Topic {c} -> {t}: {r}" for c, (t, r) in sorted(MERGE.items())]
lines += ["", "## Relabels", ""]
lines += [f"- Topic {c}: '{l}'" for c, l in sorted(RELABEL.items())]
lines += ["", "## Frozen codebook", ""]
for th, g in codebook.groupby("theme_id"):
    lines += [f"### Theme {th}: {g['theme_label'].iloc[0]}", ""]
    for r in g.itertuples():
        lines += [f"- **{r.topic_id} {r.label}** (stab {r.stability:.2f}, "
                  f"coh {r.cohesion:.2f}, w {r.weight:.0f}): {r.definition}"]
    lines += [""]
out = ROOT / "quality_reports" / "20260822_codebook_ratification.md"
out.write_text("\n".join(lines))
print(f"written: {out}")

written: /project2/njia_1089/acroh/projects/llm_method/quality_reports/20260822_codebook_ratification.md


## Verification

In [5]:
back = pd.read_parquet(TAX / "codebook_v1.parquet")
assert back["topic_id"].is_unique and len(back) == len(KEEP)
t2b = pd.read_parquet(TAX / "track2_assigned.parquet")
assert len(t2b) == 32385
print(f"1) codebook : {len(back)} topics / {back['theme_id'].nunique()} "
      f"themes, frozen {sha[:16]}…")
print(f"2) coverage : track1 {(t1['topic_id'] >= 0).mean():.1%} "
      f"(membership), track2 {(t2['topic_id'] >= 0).mean():.1%} "
      f"(centroid, tau {TAU})")
print(f"3) NEXT     : CAP evaluation + figures (11)")

1) codebook : 111 topics / 14 themes, frozen fd81e7814be41661…
2) coverage : track1 83.2% (membership), track2 92.6% (centroid, tau 0.68)
3) NEXT     : CAP evaluation + figures (11)
